# Part 2: Query structured data with Fabric IQ

Part 2 at Caldova begins when a procurement analyst finds Meridian API Works invoice and quality evidence for lot `AUR-API-7`. The lot contains Caldovexine (`SUB-001`). The analyst needs two different kinds of structured answers. First, the business **relationships**: which medicinal products contain the substance and which manufacturers make it. Second, the **performance trend**: is Meridian API Works getting better or worse at delivering on time and in full? Fabric IQ answers both from Caldova's lakehouse.

**🎯 Mission**
- Query the Caldova **ontology** through its MCP endpoint for product, substance, and manufacturer relationships
- Query the **Fabric data agent** through its MCP endpoint for supplier performance rankings and trends
- Compare the two kinds of answers and trace each one back to the lakehouse tables

**Before you run:** Follow [Exercise 2 in the lab guide](../../instructions/overview.md#exercise-2-fabric-iq) to explore the lakehouse tables, the ontology, and the data agent in Microsoft Fabric. Then follow this notebook from top to bottom, running the code cells in sequence.

## The building blocks

- **Fabric IQ ontology**: Gives business meaning to lakehouse tables as entities (such as `MedicinalProduct` and `ActiveSubstance`) and relationships (such as **contains** and **manufactures**). Its MCP endpoint exposes an `ask_ontology` tool that answers questions over those relationships.
- **Fabric data agent**: Turns natural-language questions into queries over the tables selected for it. Once published, it exposes an MCP endpoint with a tool named after the agent.
- **Delegated sign-in**: Both endpoints run as you. The notebook signs you in with `InteractiveBrowserCredential` and uses your Fabric permissions.

## Step 1: Load configuration and sign in to Fabric

Load the Fabric settings and MCP URLs written to `.env` during setup, sign in with your lab account, and define a helper that opens an authenticated MCP session for any Fabric MCP URL. The HTTP client follows redirects because Fabric routes MCP requests through its service endpoint.

If the browser sign-in window is hidden, minimize other windows **inside the virtual machine**. Keep the outer browser hosting Skillable open.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import os
from contextlib import asynccontextmanager

import httpx
from azure.identity import InteractiveBrowserCredential
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

load_dotenv(override=True)

FABRIC_TENANT_ID = os.getenv("FABRIC_TENANT_ID") or os.environ["AZURE_TENANT_ID"]
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_ONTOLOGY_ID = os.environ["FABRIC_ONTOLOGY_ID"]
FABRIC_ONTOLOGY_MCP_URL = os.getenv("FABRIC_ONTOLOGY_MCP_URL") or (
    "https://api.fabric.microsoft.com/v1/mcp/dataPlane/"
    f"workspaces/{FABRIC_WORKSPACE_ID}/items/{FABRIC_ONTOLOGY_ID}/ontologyEndpoint"
)
FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"

credential = InteractiveBrowserCredential(tenant_id=FABRIC_TENANT_ID)
credential.get_token(FABRIC_SCOPE)


@asynccontextmanager
async def open_fabric_mcp_session(mcp_url):
    token = credential.get_token(FABRIC_SCOPE)
    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token.token}"},
        follow_redirects=True,
        timeout=httpx.Timeout(30, read=300),
    ) as http_client:
        async with streamable_http_client(mcp_url, http_client=http_client) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session


print("Signed in to Fabric")

---

## A: Query the ontology through MCP

## Step 2: List the ontology tools

Open an authenticated MCP session to the `CaldovaMedicinalProductOntology` endpoint and inspect every tool it exposes, including each tool's input schema.

In [ ]:
from rich.console import Console

console = Console()

async with open_fabric_mcp_session(FABRIC_ONTOLOGY_MCP_URL) as session:
    tools_response = await session.list_tools()

for tool in tools_response.tools:
    console.print(f"[bold cyan]Tool:[/] {tool.name}")
    if tool.description:
        console.print(f"[bold]Description:[/] {tool.description}")
    console.print("[bold]Input schema:[/]")
    console.print_json(data=tool.inputSchema)
    console.print()

## Step 3: Ask the ontology about Caldovexine

Call the ontology's `ask_ontology` tool with a relationship question. The tool answers from the lakehouse data bound to the ontology and returns the answer with supporting evidence. Compare the answer with the `ProductActiveSubstance` and `ManufacturerActiveSubstance` rows you previewed in Fabric.

⏱️ Answers typically take 30 to 60 seconds. If a question takes longer than about a minute, Fabric can return an empty response, which surfaces as `Error parsing JSON response` followed by a timeout. If that happens, rerun the cell.

In [ ]:
from datetime import timedelta

from rich.markdown import Markdown

question = "Which manufacturers make Caldovexine (SUB-001), and which Caldova medicinal products contain it?"

async with open_fabric_mcp_session(FABRIC_ONTOLOGY_MCP_URL) as session:
    result = await session.call_tool(
        "ask_ontology",
        {"request": question},
        read_timeout_seconds=timedelta(seconds=120),
    )

if result.isError:
    raise RuntimeError(f"ask_ontology failed: {result.content}")
ontology_answer = "".join(block.text for block in result.content if block.type == "text")
console.print(Markdown(ontology_answer))

---

## B: Query the Fabric data agent through MCP

The ontology answers relationship questions. Trend questions, such as how a supplier's on-time-in-full (OTIF) rate changed week by week, need aggregation over the `SupplierPerformance` table. Setup created and published **CaldovaSupplierDataAgent** for that kind of question. It can query only the supplier analytics tables in the lakehouse, and its instructions explain how those tables join and how to present the metrics.

## Step 4: List the data agent tools

Open an MCP session to the published data agent and inspect its tool. A data agent exposes a single tool named after the agent, which takes a natural-language `userQuestion`.

In [ ]:
FABRIC_DATA_AGENT_MCP_URL = os.environ["FABRIC_DATA_AGENT_MCP_URL"]

async with open_fabric_mcp_session(FABRIC_DATA_AGENT_MCP_URL) as session:
    tools_response = await session.list_tools()

data_agent_tool = tools_response.tools[0].name
for tool in tools_response.tools:
    console.print(f"[bold cyan]Tool:[/] {tool.name}")
    if tool.description:
        console.print(f"[bold]Description:[/] {tool.description}")
    console.print("[bold]Input schema:[/]")
    console.print_json(data=tool.inputSchema)
    console.print()

## Step 5: Ask the data agent which suppliers are falling behind

Ask an analytical question. The data agent writes a query over its tables, runs it, and summarizes the result. The response also includes a deep link that opens the answer in Fabric, where you can inspect the generated query.

⏱️ Each answer takes about 20 to 40 seconds.

In [ ]:
async def ask_data_agent(question):
    async with open_fabric_mcp_session(FABRIC_DATA_AGENT_MCP_URL) as session:
        result = await session.call_tool(data_agent_tool, {"userQuestion": question})
    if result.isError:
        raise RuntimeError(f"{data_agent_tool} failed: {result.content}")
    answer = "\n\n".join(block.text for block in result.content if block.type == "text")
    deep_link_url = (result.structuredContent or {}).get("deepLinkUrl")
    if deep_link_url:
        answer += f"\n\n[Open this answer in Fabric]({deep_link_url})"
    console.print(Markdown(answer))


await ask_data_agent(
    "Which three suppliers have the lowest OTIF in the latest weekly snapshot, and what are their batch rejection rates?"
)

## Step 6: Follow Meridian API Works over time

Now ask about the supplier behind lot `AUR-API-7`. A trend needs several snapshots, which is the kind of aggregation the ontology does not do. Change the question to explore other suppliers or metrics.

In [ ]:
await ask_data_agent(
    "How did Meridian API Works' OTIF and batch rejection rate change over the last four weekly snapshots?"
)

#### 🔍 Source Hunt

Compare the answers from Steps 3, 5, and 6. Can you find:

1. Which answer came from **relationships** in the ontology, and which came from **aggregations** over the lakehouse tables?
2. Open a deep link from Step 5 or 6. Which tables does the generated query read, and does it use the latest `DateKey`?
3. In the lakehouse, preview `SupplierPerformance` for Meridian API Works. Do the latest values match the data agent's answer?
4. Ask the data agent which products contain Caldovexine. Does it decline, as its instructions require?

## ✅ Mission complete

**What you built:**

* ✓ **Ontology over MCP**: `ask_ontology` answered which manufacturers make Caldovexine and which products contain it, from the relationships modelled in `CaldovaMedicinalProductOntology`.
* ✓ **Data agent over MCP**: `CaldovaSupplierDataAgent` answered ranking and trend questions about supplier performance, with a deep link to the generated query.
* ✓ **Two kinds of structured answers**: Relationship facts from the ontology and aggregated metrics from the data agent, each traceable to the lakehouse tables.

Fabric IQ gives agents structured business data. The next part adds what people are saying and deciding in Microsoft 365.

➡️ Select **Next** in the lab sidebar to start the [Exercise 3 browser exploration](../../instructions/overview.md#exercise-3-work-iq), then open [Part 3: Bring in work context with Work IQ](part3-work-iq.ipynb). Outside Skillable, use the exercise link.